# **Import**

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

from joblib import dump

from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.linear_model import LinearRegression, LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    learning_curve,
    train_test_split
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from tensorflow.keras.utils import plot_model




# **Leitura do dataset**

In [ ]:
previsores = pd.read_csv("/content/features_cicid.csv")
classes = pd.read_csv("/content/class_cicids.csv")

In [ ]:
previsor_train, previsor_test, classe_train, classe_test = train_test_split(previsores, classes, test_size=0.20)

# **Gradient Boosting Classifier**

In [ ]:
gboost = GradientBoostingClassifier(random_state=42, n_estimators=50,max_depth=3, learning_rate=0.2)
gboost.fit(previsor_train, classe_train.values.ravel())

In [ ]:
previsoes_testeGB = gboost.predict(previsor_test)
acuracia = accuracy_score(classe_test, previsoes_testeGB)

In [ ]:
print(f'Acurácia do Gradient Boosting: {acuracia}')

In [ ]:
matrizGB = confusion_matrix(classe_test, previsoes_testeGB)
plt.figure(figsize=(8, 6))
sns.heatmap(matrizGB, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Gradient Boosting')
plt.savefig('matrizdeconGB.png')
plt.show()

In [ ]:
print(classification_report(classe_test, previsoes_testeGB))

In [ ]:
gboost = GradientBoostingClassifier(random_state=42, n_estimators=50,max_depth=3, learning_rate=0.2)

train_sizes, train_scores, test_scores = learning_curve(
    gboost, previsor_train, classe_train.values.ravel(), cv=10, scoring='accuracy', train_sizes=np.linspace(0.1, 1.0, 10)
)


train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validation")
plt.xlabel("Tamanho do conjunto de treinamento")
plt.ylabel("Acurácia")
plt.legend(loc="best")
plt.title("Curva de aprendizado - Gradient Boosting")
plt.savefig('curva_aprendizado_gboost.png')
plt.show()

In [ ]:
gboostpkl = joblib.load('/content/modelo_gradient_boosting.pkl')

In [ ]:



train_sizes, train_scores, test_scores = learning_curve(
    gboostpkl, previsor_train, classe_train.values.ravel(), cv=10, scoring='neg_log_loss', train_sizes=np.linspace(0.1, 1.0, 10)
)


train_scores_mean = -np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validação")
plt.xlabel("Tamanho do conjunto de treinamento")
plt.ylabel("Log Loss")
plt.legend(loc="best")
plt.title("Curva de aprendizado - Gradient Boosting (Log Loss)")
plt.savefig('curva_aprendizado_gboost_loss.png')
plt.show()


In [ ]:
gboost = joblib.load('/content/modelo_gradient_boosting.pkl')


num_folds = 10

kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)


acuracias = []


for fold, (train_index, test_index) in enumerate(kf.split(previsores, classes)):
    print(f"\nFold {fold + 1}/{num_folds}")


    previsor_train_fold, previsor_test_fold = previsores.iloc[train_index], previsores.iloc[test_index]
    classe_train_fold, classe_test_fold = classes.iloc[train_index], classes.iloc[test_index]


    gboost.fit(previsor_train_fold, classe_train_fold.values.ravel())


    previsoes_teste = gboost.predict(previsor_test_fold)


    acuracia = accuracy_score(classe_test_fold, previsoes_teste)
    acuracias.append(acuracia)

    print(f'Acurácia do Gradient Boosting no Fold {fold + 1}: {acuracia}')


media_acuracias = np.mean(acuracias)
print(f"\nMédia das Acurácias: {media_acuracias}")


# **Regressão Logistica**

In [ ]:
regressao_logistica = LogisticRegression(random_state=42, max_iter=100)
regressao_logistica.fit(previsor_train,classe_train.values.ravel())

In [ ]:
acuracia_regressao_logistica = regressao_logistica.score(previsor_test, classe_test)
print(f'Acurácia da Regressão Logística: {acuracia_regressao_logistica}')

In [ ]:
dump(regressao_logistica, 'modelo_regressao_logistica.pkl')

In [ ]:
matrRl = confusion_matrix(classe_test, previsoesLR )
plt.figure(figsize=(8, 6))
sns.heatmap(matrRl, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Regressão Logística')
plt.savefig('matrizdeconRL.png')
plt.show()

In [ ]:
print(classification_report(classe_test, previsoesLR))

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    LogisticRegression(random_state=42, max_iter=100, solver='lbfgs'),
    previsor_train,
    classe_train.values.ravel(),
    cv=10,
    scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10)
)


train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(
    train_sizes,
    train_scores_mean - train_scores_std,
    train_scores_mean + train_scores_std,
    alpha=0.1,
    color="r"
)
plt.fill_between(
    train_sizes,
    test_scores_mean - test_scores_std,
    test_scores_mean + test_scores_std,
    alpha=0.1,
    color="g"
)
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validação")
plt.xlabel("Tamanho do Conjunto de Treinamento")
plt.ylabel("Acurácia")
plt.legend(loc="best")
plt.title("Curva de Aprendizado - Regressão Logística")
plt.savefig('curva_aprendizado_acuracia_regressao_logistica.png')
plt.show()


regressao_logistica = LogisticRegression(random_state=42, max_iter=500)
regressao_logistica.fit(previsor_train, classe_train.values.ravel())
acuracia_regressao_logistica = regressao_logistica.score(previsor_test, classe_test)
print(f'Acurácia da Regressão Logística: {acuracia_regressao_logistica}')


In [ ]:

regressao_logistica = LogisticRegression(random_state=42, max_iter=100, solver='lbfgs')


train_sizes, train_scores, test_scores = learning_curve(
    regressao_logistica, previsor_train, classe_train, cv=10,
    scoring='neg_log_loss', train_sizes=np.linspace(0.1, 1.0, 10)
)


train_scores_mean = -np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std,
                 alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std,
                 alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validation")


plt.xlabel("Tamanho do conjunto de treinamento")
plt.ylabel("Loss")
plt.legend(loc="best")
plt.title("Curva de aprendizado - Regressão logística")


plt.savefig('curvaaprendizado_loss.png')
plt.show()


In [ ]:

regressao_logistica = LogisticRegression(random_state=42, max_iter=100)


num_folds = 10

kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

acuracias = []

for fold, (train_index, test_index) in enumerate(kf.split(previsores, classes)):
    print(f"\nFold {fold + 1}/{num_folds}")


    train_index = list(train_index)
    test_index = list(test_index)


    previsor_train_fold, previsor_test_fold = previsores.iloc[train_index], previsores.iloc[test_index]
    classe_train_fold, classe_test_fold = classes.iloc[train_index], classes.iloc[test_index]

    regressao_logistica.fit(previsor_train_fold, classe_train_fold.values.ravel())

    previsoes_testeRL = regressao_logistica.predict(previsor_test_fold)


    acuracia = accuracy_score(classe_test_fold, previsoes_testeRL)
    acuracias.append(acuracia)

    print(f'Acurácia da Regressão Logística no Fold {fold + 1}: {acuracia}')

media_acuracias = np.mean(acuracias)
print(f"\nMédia das Acurácias: {media_acuracias}")


# **Regressão Linear**

In [ ]:
regressao_Linear = LinearRegression()
regressao_Linear.fit(previsor_train, classe_train)

In [ ]:
previsoes_testRL_continuas = regressao_Linear.predict(previsor_test)
limiar = 0.5
previsoes_testRL = (previsoes_testRL_continuas >= limiar).astype(int)
acuracia_regressao_linear = accuracy_score(classe_test, previsoes_testRL)
print(f'Acurácia da Regressão Linear com Limiar: {acuracia_regressao_linear}')

In [ ]:

regressao_linear = LinearRegression()
num_folds = 10

kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)


acuracias = []


for fold, (train_index, test_index) in enumerate(kf.split(previsor_train, classe_train)):
    print(f"\nFold {fold + 1}/{num_folds}")


    previsor_train_fold, previsor_test_fold = previsor_train.iloc[train_index], previsor_train.iloc[test_index]
    classe_train_fold, classe_test_fold = classe_train.iloc[train_index], classe_train.iloc[test_index]


    regressao_linear.fit(previsor_train_fold, classe_train_fold)


    previsoes_testeRL = regressao_linear.predict(previsor_test_fold)


    previsoes_testeRL = (previsoes_testeRL >= 0.5).astype(int)


    acuracia = accuracy_score(classe_test_fold, previsoes_testeRL)
    acuracias.append(acuracia)

    print(f'Acurácia da Regressão Linear no Fold {fold + 1}: {acuracia}')


media_acuracias = np.mean(acuracias)
print(f"\nMédia das Acurácias: {media_acuracias}")

In [ ]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(10,),
                    activation='relu',
                    solver='adam',
                    max_iter=100,
                    random_state=42)

In [ ]:
mlp.fit(previsor_train, classe_train.values.ravel())
previsoes_testeMLP = mlp.predict(previsor_test)
acuracia_mlp = accuracy_score(classe_test, previsoes_testeMLP)
print(f'Acurácia do MLP: {acuracia_mlp}')

In [ ]:
regressao_Linear = LinearRegression()
regressao_Linear.fit(previsor_train, classe_train)

In [ ]:
previsoes_testRL_continuas = regressao_Linear.predict(previsor_test)

In [ ]:
limiar = 0.5
previsoes_testRL = (previsoes_testRL_continuas >= limiar).astype(int)

In [ ]:
acuracia_regressao_linear = accuracy_score(classe_test, previsoes_testRL)
print(f'Acurácia da Regressão Linear com Limiar: {acuracia_regressao_linear}')

In [ ]:
dump(regressao_Linear, 'modelo_regressaolinear.pkl')

In [ ]:
matrizRL = confusion_matrix(classe_test, previsoes_testRL)
plt.figure(figsize=(8, 6))
sns.heatmap(matrizRL, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Linear Regression')
plt.savefig('matrizdeconRL.png')
plt.show()

In [ ]:
print(classification_report(classe_test, previsoes_testRL))

In [ ]:
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns


previsor_train_com_constante = sm.add_constant(previsor_train)

modelo_stats = sm.OLS(classe_train, previsor_train_com_constante).fit()

print(modelo_stats.summary())

variaveis = modelo_stats.params.index
coeficientes_pvalues_df = pd.DataFrame({
    "Variável": variaveis,
    "Coeficiente": modelo_stats.params.values,
    "P-valor": modelo_stats.pvalues.values
})

coeficientes_pvalues_df["Impacto_Absoluto"] = coeficientes_pvalues_df["Coeficiente"].abs()
coeficientes_pvalues_df = coeficientes_pvalues_df.sort_values(by="Impacto_Absoluto", ascending=False).drop(columns=["Impacto_Absoluto"])

print("\nCoeficientes e P-valores:")
print(coeficientes_pvalues_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=coeficientes_pvalues_df, x="Coeficiente", y="Variável", palette="viridis")
plt.axvline(0, color="red", linestyle="--")
plt.title("Importância dos Coeficientes da Regressão Linear com P-valores")
plt.xlabel("Coeficiente")
plt.ylabel("Variável")
plt.tight_layout()
plt.savefig('coeficientes_com_pvalores.png')
plt.show()


In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.metrics import accuracy_score, make_scorer
import numpy as np
import matplotlib.pyplot as plt


accuracy_scorer = make_scorer(lambda y_true, y_pred: accuracy_score(y_true, (y_pred >= 0.5).astype(int)))


train_sizes, train_scores, test_scores = learning_curve(
    regressao_Linear, previsor_train, classe_train,
    cv=10, scoring=accuracy_scorer, train_sizes=np.linspace(0.1, 1.0, 10)
)


train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std,
                 alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std,
                 alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validation")


plt.xlabel("Training Set Size")
plt.ylabel("Accuracy")
plt.legend(loc="best")
plt.title("Learning Curve - Regressão Linear (Accuracy)")
plt.savefig('curvaaprendizadoRL_accuracy.png')
plt.show()


In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    regressao_Linear,
    previsor_train,
    classe_train,
    cv=10,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 10),
    random_state=42
)


train_scores_mean = -np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(
    train_sizes,
    train_scores_mean - train_scores_std,
    train_scores_mean + train_scores_std,
    alpha=0.1,
    color="r",
    label="Training"
)
plt.fill_between(
    train_sizes,
    test_scores_mean - test_scores_std,
    test_scores_mean + test_scores_std,
    alpha=0.1,
    color="g",
    label="Validation"
)
plt.plot(train_sizes, train_scores_mean, "o-", color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, "o-", color="g", label="Validation")
plt.xlabel("Training Set Size")
plt.ylabel("Loss (MSE)")
plt.legend(loc="best")
plt.title("Learning Curve - Linear Regression")
plt.savefig("curva_de_perda_regressao_linear.png")
plt.show()


In [ ]:

regressao_linear = LinearRegression()
num_folds = 10

kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)


acuracias = []


for fold, (train_index, test_index) in enumerate(kf.split(previsor_train, classe_train)):
    print(f"\nFold {fold + 1}/{num_folds}")


    previsor_train_fold, previsor_test_fold = previsor_train.iloc[train_index], previsor_train.iloc[test_index]
    classe_train_fold, classe_test_fold = classe_train.iloc[train_index], classe_train.iloc[test_index]


    regressao_linear.fit(previsor_train_fold, classe_train_fold)


    previsoes_testeRL = regressao_linear.predict(previsor_test_fold)


    previsoes_testeRL = (previsoes_testeRL >= 0.5).astype(int)


    acuracia = accuracy_score(classe_test_fold, previsoes_testeRL)
    acuracias.append(acuracia)

    print(f'Acurácia da Regressão Linear no Fold {fold + 1}: {acuracia}')


media_acuracias = np.mean(acuracias)
print(f"\nMédia das Acurácias: {media_acuracias}")


# **Árvore de Decição**

In [ ]:
arvore_decisao = DecisionTreeClassifier()
arvore_decisao.fit(previsor_train, classe_train)

In [ ]:
previsoes_testeAD = arvore_decisao.predict(previsor_test)

In [ ]:
acuracia = accuracy_score(classe_test, previsoes_testeAD)
print(f'Acurácia da árvore de decisão: {acuracia}')

In [ ]:
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np


arvore_decisao = DecisionTreeClassifier()


num_folds = 10


kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
df_classes = pd.DataFrame(classes, columns=['target'])
acuracias = []

for fold, (train_index, test_index) in enumerate(kf.split(previsores, classes)):
    print(f"\nFold {fold + 1}/{num_folds}")


    train_index = list(train_index)
    test_index = list(test_index)


    previsor_train_fold, previsor_test_fold = previsores.iloc[train_index], previsores.iloc[test_index]
    classe_train_fold, classe_test_fold = classes.iloc[train_index], classes.iloc[test_index]


    arvore_decisao.fit(previsor_train_fold, classe_train_fold)
    previsoes_testeAD = arvore_decisao.predict(previsor_test_fold)


    acuracia = accuracy_score(classe_test_fold, previsoes_testeAD)
    acuracias.append(acuracia)

    print(f'Acurácia da árvore de decisão no Fold {fold + 1}: {acuracia}')

media_acuracias = np.mean(acuracias)
print(f"\nMédia das Acurácias: {media_acuracias}")

In [ ]:
arvore_decisao = DecisionTreeClassifier()
arvore_decisao.fit(previsor_train, classe_train)

In [ ]:
previsoes_testeAD = arvore_decisao.predict(previsor_test)

In [ ]:
acuracia = accuracy_score(classe_test, previsoes_testeAD)
print(f'Acurácia da árvore de decisão: {acuracia}')

In [ ]:
dump(arvore_decisao, 'modelo_arvore_decisao.pkl')

In [ ]:
matrizDT = confusion_matrix(classe_test, previsoes_testeAD)
plt.figure(figsize=(8, 6))
sns.heatmap(matrizDT, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Decision Tree')
plt.savefig('matrizdeconDT.png')
plt.show()

In [ ]:
print(classification_report(classe_test, previsoes_testeAD))

In [ ]:
from sklearn.tree import plot_tree

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(arvore_decisao, filled=True, feature_names=previsor_train.columns, class_names=True, rounded=True)
plt.savefig('arvore.png')
plt.show()

In [ ]:
from sklearn.tree import export_text
feature_names = list(previsores.columns)
arvore_texto = export_text(arvore_decisao, feature_names=feature_names)
print(arvore_texto)


In [ ]:
train_sizes, train_scores, test_scores = learning_curve(arvore_decisao, previsor_train, classe_train, cv=10, scoring='neg_log_loss', train_sizes=np.linspace(0.1, 1.0, 10))


train_scores_mean = -np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validation")
plt.xlabel("Training Set Size")
plt.ylabel("Loss")
plt.legend(loc="best")
plt.title("Learning Curve - Decision Tree")
plt.savefig('curvaaprendizado.png')
plt.show()

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(arvore_decisao, previsor_train, classe_train, cv=10, scoring='accuracy', train_sizes=np.linspace(0.1, 1.0, 10))


train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validation")
plt.xlabel("Training Set Size")
plt.ylabel("Accuracy")
plt.legend(loc="best")
plt.title("Learning Curve - Decision Tree")
plt.savefig('curvaaprendizadoaccura.png')
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np


arvore_decisao = DecisionTreeClassifier()


num_folds = 10


kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
df_classes = pd.DataFrame(classes, columns=['target'])
acuracias = []

for fold, (train_index, test_index) in enumerate(kf.split(previsores, classes)):
    print(f"\nFold {fold + 1}/{num_folds}")


    train_index = list(train_index)
    test_index = list(test_index)


    previsor_train_fold, previsor_test_fold = previsores.iloc[train_index], previsores.iloc[test_index]
    classe_train_fold, classe_test_fold = classes.iloc[train_index], classes.iloc[test_index]


    arvore_decisao.fit(previsor_train_fold, classe_train_fold)
    previsoes_testeAD = arvore_decisao.predict(previsor_test_fold)


    acuracia = accuracy_score(classe_test_fold, previsoes_testeAD)
    acuracias.append(acuracia)

    print(f'Acurácia da árvore de decisão no Fold {fold + 1}: {acuracia}')

media_acuracias = np.mean(acuracias)
print(f"\nMédia das Acurácias: {media_acuracias}")


# **Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(previsor_train, classe_train.values.ravel())

In [ ]:
previsoes_testeRF = random_forest.predict(previsor_test)


In [ ]:
acuracia_rf = accuracy_score(classe_test, previsoes_testeRF)
print(f'Acurácia do Random Forest: {acuracia_rf}')

In [ ]:
dump(random_forest, 'modelo_random_forest.pkl')

In [ ]:
matrizRF = confusion_matrix(classe_test, previsoes_testeRF)
plt.figure(figsize=(8, 6))
sns.heatmap(matrizRF, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Random Forest')
plt.savefig('matrizdeconRF.png')
plt.show()

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(random_forest, previsor_train, classe_train.values.ravel(), cv=10, scoring='neg_log_loss', train_sizes=np.linspace(0.1, 1.0, 10))


train_scores_mean = -np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validation")
plt.xlabel("Training Set Size")
plt.ylabel("Loss")
plt.legend(loc="best")
plt.title("Learning Curve - Random Forest")
plt.savefig('curvaaprendizadoRF.png')
plt.show()

In [ ]:
from sklearn.tree import export_graphviz, plot_tree
import matplotlib.pyplot as plt

In [ ]:
pip install graphviz

In [ ]:
tree = random_forest.estimators_[0]

plt.figure(figsize=(20, 10))
plot_tree(tree, filled=True, feature_names=previsor_train.columns, class_names=True, fontsize=10)
plt.savefig("florestaaleatoria.png", dpi=1000, bbox_inches='tight')
plt.show()


export_graphviz(
    tree,
    out_file="tree.dot",
    feature_names=previsor_train.columns,
    filled=True,
    rounded=True,
    special_characters=True,
)


In [ ]:
from graphviz import Source
arquivo_dot = "tree.dot"
grafico = Source.from_file(arquivo_dot)
grafico.view()

# **Naive Bayes**

In [ ]:
naive_bayes = GaussianNB()
naive_bayes.fit(previsor_train, classe_train.values.ravel())

In [ ]:
previsoes_testeb = naive_bayes.predict(previsor_test)

In [ ]:
acuraciab = accuracy_score(classe_test, previsoes_testeb)
print(f'Acurácia do Naive Bayes: {acuraciab}')

In [ ]:
dump(naive_bayes, 'modelo_naive_bayes.pkl')

In [ ]:
matrizNB = confusion_matrix(classe_test, previsoes_testeb)
plt.figure(figsize=(8, 6))
sns.heatmap(matrizNB, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Naive Bayes')
plt.savefig('matrizdeconNB.png')
plt.show()

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(naive_bayes, previsor_train, classe_train.values.ravel(), cv=10, scoring='neg_log_loss', train_sizes=np.linspace(0.1, 1.0, 10))


train_scores_mean = -np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validação")
plt.xlabel("Tamanho do conjunto de treinamento")
plt.ylabel("Loss")
plt.legend(loc="best")
plt.title("Curva de aprendizado - Naive Bayes")
plt.savefig('curvaaprendizadolossnb.png')
plt.show()

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(naive_bayes, previsor_train, classe_train.values.ravel(), cv=10, scoring='accuracy', train_sizes=np.linspace(0.1, 1.0, 10))


train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validação")
plt.xlabel("Tamanho do conjunto de treinamento")
plt.ylabel("Acurácia")
plt.legend(loc="best")
plt.title("Curva de aprendizado - Naive Bayes")
plt.savefig('curvaaprendizadoaccuracynb.png')
plt.show()

In [ ]:
num_folds = 10


kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)


df_classes = pd.DataFrame(classes, columns=['target'])


acuracias = []


for fold, (train_index, test_index) in enumerate(kf.split(previsores, classes)):
    print(f"\nFold {fold + 1}/{num_folds}")


    train_index = list(train_index)
    test_index = list(test_index)


    previsor_train_fold, previsor_test_fold = previsores.iloc[train_index], previsores.iloc[test_index]
    classe_train_fold, classe_test_fold = classes.iloc[train_index], classes.iloc[test_index]


    naive_bayes.fit(previsor_train_fold, classe_train_fold.values.ravel())

    previsoes_testeb = naive_bayes.predict(previsor_test_fold)

    acuracia = accuracy_score(classe_test_fold, previsoes_testeb)
    acuracias.append(acuracia)


    print(f'Acurácia do Naive Bayes no Fold {fold + 1}: {acuracia}')


media_acuracias = np.mean(acuracias)
print(f"\nMédia das Acurácias: {media_acuracias}")

# **AdaBoost**

In [ ]:
classificador_base = DecisionTreeClassifier(max_depth=1)

In [ ]:
adaboost = AdaBoostClassifier(estimator=classificador_base, n_estimators=50, random_state=42, algorithm='SAMME')
adaboost.fit(previsor_train, classe_train.values.ravel())

In [ ]:
previsoes_testAdB = adaboost.predict(previsor_test)
acuracia = accuracy_score(classe_test, previsoes_testAdB)
print(f'Acurácia do AdaBoost: {acuracia}')

In [ ]:
from joblib import dump
dump(adaboost, 'modelo_adaboost.pkl')
print("Modelo salvo como 'modelo_adaboost.pkl'.")

In [ ]:
matrizsAdaB = confusion_matrix(classe_test, previsoes_testAdB)
plt.figure(figsize=(8, 6))
sns.heatmap(matrizsAdaB, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(' AdaBoost confusion matrix')
plt.savefig('matrizdeconAdamB.png')
plt.show()

In [ ]:


classificador_base = DecisionTreeClassifier(max_depth=1)


adaboost = AdaBoostClassifier(estimator=classificador_base, n_estimators=50, random_state=42, algorithm='SAMME')


train_sizes, train_scores, test_scores = learning_curve(
    adaboost,
    previsor_train,
    classe_train.values.ravel(),
    cv=10,
    scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validação")
plt.xlabel("Tamanho do Conjunto de Treinamento")
plt.ylabel("Acurácia")
plt.legend(loc="best")
plt.title("Curva de Aprendizado - AdaBoost")
plt.savefig('curvaaprendizadoaccuraciadaboost.png')
plt.show()



In [ ]:
classificador_base = DecisionTreeClassifier(max_depth=1)


adaboost = AdaBoostClassifier(estimator=classificador_base, n_estimators=50, random_state=42, algorithm='SAMME')


train_sizes, train_scores, test_scores = learning_curve(
    adaboost, previsor_train, classe_train.values.ravel(),
    cv=10, scoring='neg_log_loss', train_sizes=np.linspace(0.1, 1.0, 10)
)


train_scores_mean = -np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validação")
plt.xlabel("Tamanho do conjunto de treinamento")
plt.ylabel("Log Loss")
plt.legend(loc="best")
plt.title("Curva de Aprendizado - AdaBoost (Log Loss)")
plt.savefig('curva_aprendizado_adaboost_loss.png')
plt.show()


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np

classificador_base = DecisionTreeClassifier(max_depth=1)


adaboost = AdaBoostClassifier(estimator=classificador_base, n_estimators=50, random_state=42, algorithm='SAMME')

num_folds = 10

kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

df_classes = pd.DataFrame(classes, columns=['target'])

acuracias = []


for fold, (train_index, test_index) in enumerate(kf.split(previsores, classes)):
    print(f"\nFold {fold + 1}/{num_folds}")


    train_index = list(train_index)
    test_index = list(test_index)

    previsor_train_fold, previsor_test_fold = previsores.iloc[train_index], previsores.iloc[test_index]
    classe_train_fold, classe_test_fold = classes.iloc[train_index], classes.iloc[test_index]


    adaboost.fit(previsor_train_fold, classe_train_fold.values.ravel())


    previsoes_testAdB = adaboost.predict(previsor_test_fold)


    acuracia = accuracy_score(classe_test_fold, previsoes_testAdB)
    acuracias.append(acuracia)


    print(f'Acurácia do AdaBoost no Fold {fold + 1}: {acuracia}')


media_acuracias = np.mean(acuracias)
print(f"\nMédia das Acurácias: {media_acuracias}")


# **MLP**

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(10,),
                    activation='relu',
                    solver='adam',
                    max_iter=100,
                    random_state=42)

In [ ]:
mlp.fit(previsor_train, classe_train.values.ravel())
previsoes_testeMLP = mlp.predict(previsor_test)

In [ ]:
acuracia_mlp = accuracy_score(classe_test, previsoes_testeMLP)
print(f'Acurácia do MLP: {acuracia_mlp}')

In [ ]:
print(classification_report(classe_test, previsoes_testeMLP))

In [ ]:
dump(mlp, 'modelo_mlp.pkl')

In [ ]:
matrizMLP = confusion_matrix(classe_test, previsoes_testeMLP)
plt.figure(figsize=(8, 6))
sns.heatmap(matrizMLP, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('MLP')
plt.savefig('matrizdeconMLP.png')
plt.show()

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(10,),
                    activation='relu',
                    solver='adam',
                    max_iter=100,
                    random_state=42)

train_sizes, train_scores, test_scores = learning_curve(
    mlp,
    previsor_train,
    classe_train.values.ravel(),
    cv=10,
    scoring='neg_log_loss',
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_scores_mean = -np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validation")
plt.xlabel("Training Set Size")
plt.ylabel("Loss")
plt.legend(loc="best")
plt.title("Learning Curve - MLP")
plt.savefig('curvaaprendizadoMLPNEW.png')
plt.show()

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(10,),
                    activation='relu',
                    solver='adam',
                    max_iter=100,
                    random_state=42)


train_sizes, train_scores, test_scores = learning_curve(
    mlp,
    previsor_train,
    classe_train.values.ravel(),
    cv=10,
    scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std,
                 alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std,
                 alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validação")

plt.xlabel("Tamanho do conjunto de treinamento")
plt.ylabel("Acurácia")
plt.legend(loc="best")
plt.title("Curva de aprendizado - MLP (Acurácia)")
plt.savefig('curva_aprendizado_mlp_accuracyNEW.png')
plt.show()

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(10,),
                    activation='relu',
                    solver='adam',
                    max_iter=100,
                    random_state=42)


num_folds = 10

kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

acuracias_mlp = []

for fold, (train_index, test_index) in enumerate(kf.split(previsor_train, classe_train)):
    print(f"\nFold {fold + 1}/{num_folds}")

    previsor_train_fold, previsor_test_fold = previsor_train.iloc[train_index], previsor_train.iloc[test_index]
    classe_train_fold, classe_test_fold = classe_train.iloc[train_index], classe_train.iloc[test_index]

    mlp.fit(previsor_train_fold, classe_train_fold.values.ravel())

    previsoes_testeMLP = mlp.predict(previsor_test_fold)

    acuracia_mlp = accuracy_score(classe_test_fold, previsoes_testeMLP)
    acuracias_mlp.append(acuracia_mlp)

    print(f'Acurácia do MLP no Fold {fold + 1}: {acuracia_mlp}')

media_acuracias_mlp = np.mean(acuracias_mlp)
print(f"\nMédia das Acurácias do MLP: {media_acuracias_mlp}")

# **SGD**

In [ ]:
sgd_model = SGDClassifier(loss='log_loss', max_iter=500, tol=0.01, random_state=42)
sgd_model.fit(previsor_train, classe_train.values.ravel())

In [ ]:
previsoes_teste_sgd = sgd_model.predict(previsor_test)

In [ ]:
acuracia_sgd = accuracy_score(classe_test, previsoes_teste_sgd)
print(f'Acurácia do Stochastic Gradient Descent (SGD): {acuracia_sgd}')

In [ ]:
matrizSGD = confusion_matrix(classe_test, previsoes_teste_sgd)
plt.figure(figsize=(8, 6))
sns.heatmap(matrizSGD, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Stochastic Gradient Descent')
plt.savefig('matrizdeconSGD.png')
plt.show()

In [ ]:
print(classification_report(classe_test, previsoes_teste_sgd))

In [ ]:
dump(sgd_model,'modelo_Sgd.pkl')

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    sgd_model,
    previsor_train,
    classe_train.values.ravel(),
    cv=10,
    scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10),
)


train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)


plt.figure(figsize=(10, 6))


plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std,
                 alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std,
                 alpha=0.1, color="g")


plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validation")

plt.xlabel("Training Set Size")
plt.ylabel("Accuracy")
plt.legend(loc="best")
plt.title("Learning Curve - Stochastic Gradient Descent (SGD) ")
plt.savefig('learning_curveStochastic Gradient Descentaccuracy.png')
plt.show()

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    sgd_model, previsor_train, classe_train.values.ravel(), cv=10, scoring='neg_log_loss', train_sizes=np.linspace(0.1, 1.0, 10)
)

train_scores_mean = -np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Validation")
plt.xlabel("Training set size")
plt.ylabel("Loss")
plt.legend(loc="best")
plt.title("Learning Curve - Stochastic Gradient Descent (SGD)")
plt.savefig('curva_aprendizadoStochastic Gradient Descent_loss.png')
plt.show()

In [ ]:
num_folds = 10

kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)


acuracias = []


for fold, (train_index, test_index) in enumerate(kf.split(previsores, classes)):
    print(f"\nFold {fold + 1}/{num_folds}")


    previsor_train_fold, previsor_test_fold = previsores.iloc[train_index], previsores.iloc[test_index]
    classe_train_fold, classe_test_fold = classes.iloc[train_index], classes.iloc[test_index]


    sgd_model.fit(previsor_train_fold, classe_train_fold.values.ravel())


    previsoes_teste = sgd_model.predict(previsor_test_fold)


    acuracia = accuracy_score(classe_test_fold, previsoes_teste)
    acuracias.append(acuracia)

    print(f'Acurácia do SGD no Fold {fold + 1}: {acuracia}')
    media_acuracias = np.mean(acuracias)
    print(media_acuracias)